# Analise e proposta de padronizacao da camada trusted

Este notebook inspeciona os Parquet da camada raw e documenta as mudancas aplicadas pelo transformador. A analise nao altera nenhum dado.

In [ ]:
from pathlib import Path
import re
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_ROOT = PROJECT_ROOT / "data" / "raw"
PARQUET_FILES = sorted(RAW_ROOT.glob("season=*/round=*/session=*/*.parquet"))
print(f"Arquivos encontrados: {len(PARQUET_FILES)}")
print(f"Particoes: {len({path.parent for path in PARQUET_FILES})}")

In [ ]:
def snake_case(name: str) -> str:
    name = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", name)
    name = re.sub(r"(?<=[A-Z])(?=[A-Z][a-z])", "_", name)
    return re.sub(r"[^a-zA-Z0-9]+", "_", name).strip("_").lower()


profile_rows = []
for path in PARQUET_FILES:
    frame = pd.read_parquet(path)
    for column in frame.columns:
        series = frame[column]
        profile_rows.append(
            {
                "table": path.stem,
                "column": column,
                "standard_name": snake_case(column),
                "dtype": str(series.dtype),
                "nullable": bool(series.isna().any()),
                "null_count": int(series.isna().sum()),
                "example_values": series.dropna().astype(str).drop_duplicates().head(3).tolist(),
            }
        )
profile = pd.DataFrame(profile_rows).drop_duplicates(["table", "column"])
profile.sort_values(["table", "column"])

In [ ]:
schema_rows = []
for path in PARQUET_FILES:
    frame = pd.read_parquet(path)
    schema_rows.extend((path.stem, column, str(frame[column].dtype)) for column in frame.columns)
schema_check = pd.DataFrame(schema_rows, columns=["table", "column", "dtype"]).drop_duplicates()
schema_check.groupby(["table", "column"], as_index=False).size().query("size > 1")

In [ ]:
normalization_rules = pd.DataFrame(
    [
        ("nomes", "Todas as colunas", "snake_case, sem espacos e sem maiusculas"),
        ("tipos", "Identificadores", "string pandas, preservando zeros e nulos"),
        ("tipos", "Contagens e posicoes", "Int64 nullable quando a origem e numerica inteira"),
        ("timestamps", "Date e LapStartDate", "datetime64[ns, UTC], assumindo timestamps raw em UTC"),
        ("duracoes", "Campos timedelta", "float64 em milissegundos com sufixo _ms"),
        ("unidades", "Speed* e Speed", "km/h; Distance* e coordenadas em metros; temperaturas em C; humidade em percentual"),
        ("identificadores", "driver, driver_id, team_id e abreviacoes", "texto aparado; codigos em maiusculas quando aplicavel"),
    ],
    columns=["categoria", "escopo", "regra"],
)
normalization_rules

In [ ]:
sample = pd.read_parquet(PARQUET_FILES[0])
print("Amostra:", PARQUET_FILES[0])
print("Colunas timedelta:", sample.select_dtypes(include=["timedelta"]).columns.tolist())
print("Colunas timestamp:", sample.select_dtypes(include=["datetime"]).columns.tolist())
print("Nomes apos snake_case:", [snake_case(column) for column in sample.columns])

## Verificacoes de qualidade

As verificacoes abaixo geram percentual de nulos e identificam duplicidades, tempos de volta invalidos, pilotos desconhecidos, voltas sem telemetria e timestamps fora do intervalo esperado.

In [ ]:
import sys

TRUSTED_ROOT = PROJECT_ROOT / "data" / "trusted"
sys.path.insert(0, str(PROJECT_ROOT))
from src.transformation.validate_trusted import validate_dataset

quality_report = validate_dataset(TRUSTED_ROOT)
quality_summary = quality_report.groupby("check", as_index=False).agg(
    checks=("check", "size"),
    failures=("passed", lambda values: int((~values).sum())),
)
quality_summary

In [ ]:
quality_report.query("check == 'null_percentage'").sort_values("violation_pct", ascending=False).head(20)

quality_report.query("not passed").sort_values(["check", "violation_pct"], ascending=[True, False])